# LA Studio forced-alignment — MMS Forced Aligner ONNX

This notebook loads exactly `mms-forced-aligner-onnx` (`onnx-community/mms-300m-1130-forced-aligner-ONNX`) on CUDA.
It does not use API Gateway and refuses every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio.


In [ ]:
!nvidia-smi
%pip install -q "onnxruntime-gpu==1.22.0" "transformers==4.57.6" "huggingface-hub==0.36.0" "git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git@11855d1de76af2b490dd2e8e2db2661805ae90a0" "fastapi==0.115.12" "uvicorn==0.34.3" "python-multipart==0.0.20"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_alignment_worker.py')
WORKER.write_text('import os\nimport re\nimport subprocess\nimport tempfile\nimport threading\nfrom pathlib import Path\n\nimport torch\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nimport math\nimport numpy as np\nimport onnxruntime as ort\nfrom huggingface_hub import snapshot_download\nfrom transformers import AutoConfig, AutoTokenizer\nfrom ctc_forced_aligner import get_alignments, get_spans, postprocess_results, preprocess_text\n\nMODEL_ID = "mms-forced-aligner-onnx"\nMODEL_NAME = "MMS Forced Aligner ONNX"\nUPSTREAM_MODEL = "onnx-community/mms-300m-1130-forced-aligner-ONNX"\nSUPPORTED_LANGUAGES = ["158 languages (ISO 639-1 or ISO 639-3)"]\nMODEL_DIR = snapshot_download(\n    UPSTREAM_MODEL,\n    allow_patterns=[\n        "config.json", "preprocessor_config.json", "special_tokens_map.json",\n        "tokenizer.json", "tokenizer_config.json", "vocab.json", "onnx/model_fp16.onnx",\n    ],\n)\nSESSION = ort.InferenceSession(\n    str(Path(MODEL_DIR) / "onnx/model_fp16.onnx"),\n    providers=["CUDAExecutionProvider"],\n)\nif "CUDAExecutionProvider" not in SESSION.get_providers():\n    raise RuntimeError("MMS ONNX did not activate CUDAExecutionProvider")\nTOKENIZER = AutoTokenizer.from_pretrained(MODEL_DIR, word_delimiter_token=None)\nCONFIG = AutoConfig.from_pretrained(MODEL_DIR)\nRATIO = int(CONFIG.inputs_to_logits_ratio)\nSAMPLE_RATE = 16000\n\nISO3 = {\n    "ar": "ara", "be": "bel", "bg": "bul", "de": "deu", "el": "ell", "en": "eng",\n    "fa": "fas", "he": "heb", "kk": "kaz", "ky": "kir", "lv": "lav", "lt": "lit",\n    "mk": "mkd", "mn": "mon", "ru": "rus", "sr": "srp", "th": "tha", "tr": "tur",\n    "ug": "uig", "uk": "ukr", "yi": "yid", "vi": "vie", "zh": "chi", "ja": "jpn",\n    "fr": "fra", "es": "spa", "it": "ita", "pt": "por", "ko": "kor",\n}\n\ndef _load_mono(path: str):\n    raw = subprocess.run(\n        ["ffmpeg", "-nostdin", "-threads", "0", "-i", path, "-f", "f32le",\n         "-ac", "1", "-ar", str(SAMPLE_RATE), "-"],\n        check=True, capture_output=True,\n    ).stdout\n    return np.frombuffer(raw, dtype=np.float32).copy()\n\ndef _onnx_emissions(audio: np.ndarray):\n    window = 30 * SAMPLE_RATE\n    context = 2 * SAMPLE_RATE\n    context_frames = context // RATIO\n    window_frames = window // RATIO\n    if audio.size < window:\n        chunks = [audio]\n        extension = 0\n        use_context = False\n    else:\n        count = math.ceil(audio.size / window)\n        extension = count * window - audio.size\n        padded = np.pad(audio, (context, context + extension))\n        chunks = [padded[index * window:index * window + window + 2 * context] for index in range(count)]\n        use_context = True\n    outputs = []\n    input_name = SESSION.get_inputs()[0].name\n    for chunk in chunks:\n        logits = SESSION.run(None, {input_name: chunk[None, :].astype(np.float32)})[0][0]\n        if use_context:\n            logits = logits[context_frames:context_frames + window_frames]\n        outputs.append(logits)\n    emissions = np.concatenate(outputs, axis=0)\n    if extension:\n        emissions = emissions[:-(extension // RATIO)]\n    values = torch.from_numpy(emissions).float().log_softmax(-1)\n    values = torch.cat([values, torch.zeros(values.size(0), 1)], dim=1)\n    return values, RATIO * 1000.0 / SAMPLE_RATE\n\ndef align_exact(source_path: str, transcript: str, language: str):\n    iso = ISO3.get(language, language)\n    emissions, stride = _onnx_emissions(_load_mono(source_path))\n    tokens, text_tokens = preprocess_text(transcript, romanize=True, language=iso)\n    segments, scores, blank = get_alignments(emissions, tokens, TOKENIZER)\n    spans = get_spans(tokens, segments, blank)\n    return postprocess_results(text_tokens, spans, stride, scores)\n\nTOKEN = os.environ["LA_STUDIO_COLAB_ALIGNMENT_TOKEN"]\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\nMAX_AUDIO_SECONDS = 300\nALLOWED_CONTENT_TYPES = {\n    "audio/wav", "audio/x-wav", "audio/mpeg", "audio/mp4", "audio/webm",\n    "audio/ogg", "audio/flac", "application/octet-stream",\n}\nALLOWED_EXTENSIONS = {".wav", ".mp3", ".m4a", ".mp4", ".webm", ".ogg", ".flac"}\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\nMODEL_LOCK = threading.Lock()\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. Open the notebook for the selected model.",\n        )\n\ndef media_duration_seconds(path: str) -> float:\n    probe = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "default=nokey=1:noprint_wrappers=1", path],\n        text=True, capture_output=True,\n    )\n    try:\n        duration = float(probe.stdout.strip())\n    except ValueError:\n        duration = 0.0\n    if probe.returncode != 0 or duration <= 0.0:\n        raise HTTPException(status_code=415, detail="audio is unsupported or could not be decoded")\n    return duration\n\ndef validate_segments(raw_segments) -> list[dict]:\n    segments = []\n    previous_end = 0.0\n    for raw in raw_segments:\n        text = str(raw.get("text", "")).strip()\n        if not text:\n            continue\n        start = float(raw.get("start", raw.get("start_time", 0.0)))\n        end = float(raw.get("end", raw.get("end_time", start)))\n        score = max(0.0, min(1.0, float(raw.get("score", raw.get("confidence", 1.0)))))\n        if start < 0.0 or end < start or start + 0.002 < previous_end:\n            raise RuntimeError("aligner returned non-monotonic timestamps")\n        segments.append({"text": text, "start": start, "end": end, "score": score, "kind": raw.get("kind", "word")})\n        previous_end = end\n    if not segments:\n        raise RuntimeError("aligner returned no timestamped tokens")\n    return segments\n\napp = FastAPI(title=f"LA Studio Alignment - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "forced-alignment",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "max_audio_seconds": MAX_AUDIO_SECONDS,\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/alignments")\nasync def align(\n    audio: UploadFile = File(...),\n    transcript: str = Form(...),\n    language: str = Form("en"),\n    model: str = Form(...),\n    authorization: str | None = Header(default=None),\n):\n    authorize(authorization)\n    require_exact_model(model)\n    text = transcript.strip()\n    if not text:\n        raise HTTPException(status_code=422, detail="transcript is required")\n    suffix = Path(audio.filename or "audio.wav").suffix.lower() or ".wav"\n    if suffix not in ALLOWED_EXTENSIONS:\n        raise HTTPException(status_code=415, detail="unsupported audio filename extension")\n    if audio.content_type and audio.content_type not in ALLOWED_CONTENT_TYPES:\n        raise HTTPException(status_code=415, detail="unsupported audio MIME type")\n    if not REQUEST_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab alignment worker is busy; retry shortly")\n    source_path = None\n    try:\n        descriptor, source_path = tempfile.mkstemp(suffix=suffix)\n        with os.fdopen(descriptor, "wb") as output:\n            while chunk := await audio.read(1024 * 1024):\n                output.write(chunk)\n                if output.tell() > MAX_UPLOAD_BYTES:\n                    raise HTTPException(status_code=413, detail="audio exceeds 512 MB upload limit")\n        duration = media_duration_seconds(source_path)\n        if duration > MAX_AUDIO_SECONDS:\n            raise HTTPException(status_code=413, detail="audio exceeds the five minute duration limit")\n        with MODEL_LOCK:\n            raw_segments = align_exact(source_path, text, language.strip().lower() or "en")\n        segments = validate_segments(raw_segments)\n        return {"duration": duration, "segments": segments, "unaligned_tokens": []}\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} alignment failed: {type(error).__name__}: {str(error)[:300]}",\n        ) from error\n    finally:\n        if source_path:\n            Path(source_path).unlink(missing_ok=True)\n        await audio.close()\n        REQUEST_SLOTS.release()\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'mms-forced-aligner-onnx'
# LA Studio worker launch contract: launch-2026-08-06.1
import json
import os
import queue
import re
import secrets
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'Forced Alignment'
MODEL_ID = 'mms-forced-aligner-onnx'
PORT = 3923
TOKEN_ENV = 'LA_STUDIO_COLAB_ALIGNMENT_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_ALIGNMENT_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_ALIGNMENT_MODEL'
WORKER_LOG = Path('/content/la_studio_alignment_worker.log')
WORKER_MODULE = 'la_studio_alignment_worker'
WORKER_PYTHON = sys.executable
WORKER_PYTHON_ISOLATED = False
WORKER_ENVIRONMENT = {}
REQUIRES_CUDA = True
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def process_cmdline(pid: int) -> str:
    """Read a Linux process command line without depending on psutil."""
    try:
        return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(
            "utf-8", errors="replace"
        ).strip()
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return ""


def all_processes():
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        pid = int(entry.name)
        command = process_cmdline(pid)
        if command:
            yield pid, command


def listening_processes(port: int) -> dict[int, str]:
    """Return PIDs listening on a local TCP port via /proc socket ownership."""
    target_port = f"{port:04X}"
    socket_inodes = set()
    for table_name in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(table_name).read_text(encoding="utf-8").splitlines()[1:]
        except FileNotFoundError:
            continue
        for line in lines:
            fields = line.split()
            if len(fields) < 10:
                continue
            local_address, state, inode = fields[1], fields[3], fields[9]
            if state == "0A" and local_address.rsplit(":", 1)[-1].upper() == target_port:
                socket_inodes.add(inode)
    if not socket_inodes:
        return {}

    listeners = {}
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        try:
            descriptors = (entry / "fd").iterdir()
        except (FileNotFoundError, PermissionError):
            continue
        for descriptor in descriptors:
            try:
                target = os.readlink(descriptor)
            except (FileNotFoundError, PermissionError, OSError):
                continue
            match = re.fullmatch(r"socket:\\[(\\d+)\\]", target)
            if match and match.group(1) in socket_inodes:
                pid = int(entry.name)
                listeners[pid] = process_cmdline(pid)
                break
    return listeners


def stop_pid(pid: int) -> None:
    if pid == os.getpid():
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.2)
    try:
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass


def reclaim_previous_la_studio_worker() -> None:
    """Stop only an older LA Studio worker/tunnel for this exact local port.

    Re-running a Colab cell keeps child processes alive.  The previous launch
    created a new token but aborted before it could replace the old worker,
    forcing users to destroy the whole GPU runtime.  We identify ownership by
    the exact generated module name and never terminate a foreign listener.
    """
    stopped = []
    for pid, command in listening_processes(PORT).items():
        if WORKER_MODULE in command and "uvicorn" in command:
            stop_pid(pid)
            stopped.append(f"worker PID {pid}")

    endpoint = f"http://127.0.0.1:{PORT}"
    for pid, command in all_processes():
        if ("cloudflared" in command and "tunnel" in command and endpoint in command):
            stop_pid(pid)
            stopped.append(f"tunnel PID {pid}")

    deadline = time.monotonic() + 12
    while port_is_occupied(PORT) and time.monotonic() < deadline:
        time.sleep(0.2)
    if stopped:
        print("Stopped previous LA Studio " + ", ".join(stopped) + ".")

    if port_is_occupied(PORT):
        listeners = listening_processes(PORT)
        foreign_pids = sorted(listeners) or ["unknown"]
        raise RuntimeError(
            f"Port {PORT} is occupied by a process that is not the previous LA Studio "
            f"{CAPABILITY_LABEL} worker (PID(s): {', '.join(map(str, foreign_pids))}). "
            "Choose a fresh Colab runtime rather than terminating an unrelated process."
        )


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


reclaim_previous_la_studio_worker()

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
env.update(WORKER_ENVIRONMENT)
if WORKER_PYTHON_ISOLATED:
    # Do not let Colab's global site-packages or a notebook-level PYTHONPATH
    # bleed into a dedicated worker virtual environment.
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [WORKER_PYTHON, "-m", "uvicorn", 'la_studio_alignment_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    worker_kind = "exact CUDA" if REQUIRES_CUDA else "dedicated Colab CPU"
    print(f"Starting {worker_kind} {CAPABILITY_LABEL} worker.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower()
                        == ("cuda" if REQUIRES_CUDA else "colab-cpu")
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print(worker_kind.title() + " worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print(f"Waiting for the {worker_kind} worker...", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
